In [0]:
df_silver_products=spark.read.format("delta").load("abfss://silver@aze2estorage.dfs.core.windows.net/products")
df_silver_product_categories=spark.read.format("delta").load("abfss://silver@aze2estorage.dfs.core.windows.net/product_categories")
df_silver_product_subcategories=spark.read.format("delta").load("abfss://silver@aze2estorage.dfs.core.windows.net/product_subcategories")

In [0]:
# =========================================================================
# GOLD LAYER: dim_products Worker Notebook
# =========================================================================

# 1. Configuration & Path Definitions
GOLD_STORAGE_PATH = "abfss://gold@aze2estorage.dfs.core.windows.net/dimensions/dim_products"
TABLE_NAME = "gold.dim_products"

print(f"🔄 Starting build for {TABLE_NAME}...")
print(f"📂 Target Storage Path: {GOLD_STORAGE_PATH}")

# 2. Read Silver Source Tables
df_products = spark.read.format("delta").load("abfss://silver@aze2estorage.dfs.core.windows.net/products")
df_subcategories =spark.read.format("delta").load("abfss://silver@aze2estorage.dfs.core.windows.net/product_subcategories")
df_categories =spark.read.format("delta").load("abfss://silver@aze2estorage.dfs.core.windows.net/product_categories")

print(f"📥 Read records -> Products: {df_products.count()}, Subcategories: {df_subcategories.count()}, Categories: {df_categories.count()}")

# 3. Denormalize Product Hierarchy (Products -> Subcategories -> Categories)
dim_products = (
    df_products
    .join(
        df_subcategories,
        df_products["product_subcategory_key"] == df_subcategories["product_subcategory_key"],
        "left"
    )
    .join(
        df_categories,
        df_subcategories["product_category_key"] == df_categories["product_category_key"],
        "left"
    )
    # Select and structure the final denormalized schema for Power BI
    .select(
        df_products["product_key"],
        df_products["product_sku"],
        df_products["product_name"],
        df_products["model_name"],
        df_products["product_description"],
        df_products["color"],
        df_products["size"],
        df_products["product_style"],
        df_products["product_cost"],
        df_products["product_price"],
        df_products["profit_margin_amount"],
        df_products["profit_margin_pct"],
        df_subcategories["subcategory_name"],
        df_categories["category_name"]
    )
)

# 4. Write Delta Files to Explicit Gold Storage Path
dim_products.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(GOLD_STORAGE_PATH)

print("💾 Successfully wrote Delta files to storage path.")

# 5. Register/Update External Table in Unity Catalog
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TABLE_NAME}
    USING DELTA
    LOCATION '{GOLD_STORAGE_PATH}'
""")

print(f"🔗 Registered external table: {TABLE_NAME}")

# 6. Performance Optimization for Power BI Joins
spark.sql(f"OPTIMIZE {TABLE_NAME} ZORDER BY (product_key)")

print(f"⚡ Successfully optimized {TABLE_NAME} with Z-Ordering on product_key.")
print(f"🎉 {TABLE_NAME} build complete!")